[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/05_%E7%94%9F%E5%AD%98%E6%99%82%E9%96%93%E8%A7%A3%E6%9E%90_%E3%83%81%E3%83%A3%E3%83%BC%E3%83%B3%E5%88%86%E6%9E%90.ipynb)

In [ ]:
# === ① セットアップ（最初に実行）: 教材用の契約継続データを作ります ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(7)
n = 500
プラン = rng.choice(['月額','年額'], n, p=[0.6, 0.4])
月間利用 = rng.poisson(8, n) + 1                                  # 月あたりの利用回数
hazard = 0.04 * np.where(プラン=='月額', 1.7, 0.7) * np.exp(-0.06*(月間利用-8))  # 解約しやすさ
tenure = rng.exponential(1/hazard)                               # 解約までの月数（真の値）
打ち切り = 24.0                                                   # 観測は24か月で打ち切り
解約 = (tenure <= 打ち切り).astype(int)                           # 1=解約を観測 / 0=まだ継続(打ち切り)
継続月数 = np.minimum(tenure, 打ち切り).round(1)
surv = pd.DataFrame({'プラン': プラン, '月間利用': 月間利用, '継続月数': 継続月数, '解約': 解約})
print('準備OK（教材用の合成データ）  解約観測:', int(解約.sum()), '/ 打ち切り:', int((解約==0).sum()))

# 発展マーケ-05. 生存時間解析（カプランマイヤー・Cox）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務／データサイエンス）**。statsmodels・scipy を使います（Colabは導入済み。ローカルは `uv add statsmodels scipy`）。

**舞台設定**：サブスク顧客が「契約からどれくらいで解約するか（チャーン）」を分析したい。やっかいなのは、**まだ解約していない顧客**（＝結末が分からない）が大勢いること。これを正しく扱います。

**使う統計（読む=準1級／本質=1級）**：打ち切り・カプランマイヤー生存曲線・Cox比例ハザード。

### 📋 使うデータ：教材用の合成データ `surv`（契約500件）
`継続月数`＝契約から現在までの月数、`解約`＝1:解約した/0:まだ継続中(打ち切り)、`プラン`（月額/年額）・`月間利用`（利用回数）。

## 🎯 この章でできるようになること
- **打ち切り（censoring）** がなぜ普通の平均・回帰を壊すか説明できる
- **カプランマイヤー法**で生存曲線（継続率）を描ける
- **Cox比例ハザード**で「何が解約を早めるか」をハザード比で読める

> **前提**：ロジスティック回帰（発展01）　/　**所要**：約30分　/　**レベル**：発展（読む準1級・本質1級）

## 1. なぜ「平均継続月数」ではダメか（打ち切り）

まだ解約していない顧客の `継続月数` は「最終的な寿命」ではなく「今のところ」の値です。これを混ぜて平均すると、継続期間を **過小評価** します。

In [ ]:
naive = surv['継続月数'].mean()
churned_only = surv[surv['解約']==1]['継続月数'].mean()
print(f'全部を平均（打ち切りを無視）  : {naive:.1f} か月  ← 過小評価')
print(f'解約した人だけの平均          : {churned_only:.1f} か月  ← これも偏る')
print('→ 打ち切りを正しく扱う方法が必要')

## 2. カプランマイヤー生存曲線

各時点で「直前まで継続していた人のうち、何割が解約せず生き残ったか」を掛け合わせて、**継続率（生存率）**の曲線を作ります。打ち切りも正しく使えます。

In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight
sf = SurvfuncRight(surv['継続月数'], surv['解約'])
fig, ax = plt.subplots(figsize=(7,4.5))
ax.step(sf.surv_times, sf.surv_prob, where='post')
ax.set_xlabel('継続月数'); ax.set_ylabel('継続率（生存率）'); ax.set_ylim(0,1)
ax.set_title('カプランマイヤー曲線（全体）'); ax.grid(alpha=.3); plt.show()
# 中央生存期間（継続率が0.5を下回る月）
import numpy as np
med = sf.surv_times[np.searchsorted(-sf.surv_prob, -0.5)] if sf.surv_prob.min()<0.5 else None
print('中央生存期間 ≈', med, 'か月' if med else '（24か月内で半数は解約せず）')

## 3. プラン別に比べる

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for pl in ['月額','年額']:
    sub = surv[surv['プラン']==pl]
    s = SurvfuncRight(sub['継続月数'], sub['解約'])
    ax.step(s.surv_times, s.surv_prob, where='post', label=pl)
ax.set_xlabel('継続月数'); ax.set_ylabel('継続率'); ax.set_ylim(0,1)
ax.legend(); ax.set_title('プラン別の継続率'); ax.grid(alpha=.3); plt.show()
print('→ 年額のほうが継続率が高い（解約しにくい）')

## 4. Cox比例ハザード：何が解約を早めるか

**ハザード比(HR)** = exp(係数)。HR>1 なら解約を**早める**要因、HR<1 なら**引き止める**要因です。

In [ ]:
from statsmodels.duration.hazard_regression import PHReg
surv['月額'] = (surv['プラン']=='月額').astype(int)
model = PHReg(surv['継続月数'], surv[['月額','月間利用']], status=surv['解約']).fit()
HR = np.exp(model.params)
print('ハザード比(HR):')
print(HR.round(3))
print('\n月額: HR>1 → 年額より解約が早い / 月間利用: HR<1 → 使うほど解約しにくい')

## 🧭 まとめ
- **打ち切り**（まだ解約していない顧客）を捨てると継続期間を見誤る。
- **カプランマイヤー**は打ち切りを正しく使って継続率の曲線を描ける。
- **Cox比例ハザード**は「何が解約を早める/遅らせるか」を**ハザード比**で定量化する。

> ⚠️ **よくある間違い**
> - 打ち切り顧客を「脱落」として削除すると、長く続く人を捨てることになり **バイアス**が出る。
> - Coxは「ハザード比が時間によらず一定」という **比例ハザード仮定** を置く。崩れる場合は層別などが必要。
> - 生存曲線は**階段状**（イベントが起きた時点だけ下がる）。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
# Q1: Cox係数が 0.69 のときのハザード比 exp(係数) を ans に
ans = None   # 例: np.exp(0.69)
_check('Q1 ハザード比', ans, np.exp(0.69))

In [ ]:
import numpy as np
# Q2: 係数が 0（効果なし）のときのハザード比を ans に
ans = None   # ヒント: exp(0)
_check('Q2 効果なしのHR', ans, np.exp(0))

In [ ]:
# Q3: 20人中5人が最初の月に解約。カプランマイヤーの『1か月生存率』(20−5)/20 を ans に
ans = None   # 例: (20-5)/20
_check('Q3 1か月生存率', ans, (20-5)/20)

---
## 🏆 練習問題

**問1.** `月間利用` を「少（≤5）/多（>5）」の2群に分けて、カプランマイヤー曲線を比べよう。

**問2.** Coxに `月間利用` を入れたときのハザード比から、利用が1回増えるごとの解約リスク変化を読み取ろう。

**問3.**（考察）打ち切りが全体の何割か数え、もし打ち切りを削除して平均したら継続期間はどう偏るか説明しよう。

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/05_%E7%94%9F%E5%AD%98%E6%99%82%E9%96%93%E8%A7%A3%E6%9E%90_%E3%83%81%E3%83%A3%E3%83%BC%E3%83%B3%E5%88%86%E6%9E%90.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 打ち切り | 観測終了時にまだイベント未発生 |
| 生存関数 S(t) | 時点tまで継続している確率 |
| カプランマイヤー | 打ち切り対応の生存曲線推定 |
| ハザード | その瞬間に解約する率 |
| ハザード比 HR | exp(係数)。>1で解約を早める |

```python
from statsmodels.duration.survfunc import SurvfuncRight
sf = SurvfuncRight(time, event); sf.surv_times, sf.surv_prob
from statsmodels.duration.hazard_regression import PHReg
PHReg(time, X, status=event).fit().params   # exp() でハザード比
```